In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable
from langchain_google_genai import ChatGoogleGenerativeAI

# large_model = init_chat_model("claude-sonnet-4-5")
# standard_model = init_chat_model("gpt-5-nano")
large_model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview")
standard_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model=standard_model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

Oh, the office plant! Let me check my to-do list... Yep, it says "Water the plant" right here. I'll go do that right now! Thanks for reminding me!


In [6]:
print(response["messages"][-1].response_metadata["model_name"])

gemini-2.5-flash-lite


In [7]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'That\'s a good question! I was actually looking at the roots earlier, and I noticed a few poking out of the drainage holes at the bottom. \n\nSince it’s been growing so quickly this week, I think we should probably move it to a pot about an inch or two wider in the next couple of weeks. Would you like me to add "buy a new pot" to our supply list for the next run, or should I see if we have an extra one stored in the supply closet first?', 'extras': {'signature': 'EjQKMgG+Pvb7dpNZ44Gi0pS5OoEJ5v/HklA6VtWhm3vrLiff/HHx/DyY8Z1ebOMf2ypNeaid'}}]


In [8]:
print(response["messages"][-1].response_metadata["model_name"])

gemini-3.1-flash-lite-preview
